# HT-9046MX Shared LSTM Autoencoder (Google Colab)

Notebook นี้เทรน **LSTM Autoencoder กลาง 1 โมเดล** จากหลาย handler/module แต่เก็บ **scaler และ anomaly threshold แยกตาม `machine_id + module_id`** เพื่อไม่ให้ความต่างของ baseline แต่ละเครื่องถูกตีความเป็น anomaly

## Goal

- ใช้เฉพาะช่วง `Status_n=On`, `Busy_n=0`
- ตัด `ChangeValve`, `AdjustValve`, `MValveHome`, Valve ติดลบ และค่าอุณหภูมิ sentinel `-200`
- ไม่สร้าง window ข้าม transition หรือช่องว่างเวลา
- เทรน shared model ด้วยข้อมูลที่ normalize แยกแต่ละ machine-module
- calibrate threshold และรายงาน test metrics แยกแต่ละ machine-module
- export model, scalers, thresholds, metrics และ inference CSV ไปยัง Google Drive


## Setup

1. ใน Colab เลือก **Runtime → Change runtime type → T4 GPU**
2. Clone/ดาวน์โหลด repo `Bookkapp/ht9046mx-shared-pdm-colab` ไว้ที่ `MyDrive/Data Analysis`
3. คัดลอกเฉพาะโฟลเดอร์ raw data ทั้ง 6 เครื่องเข้า `MyDrive/Data Analysis`; raw logs จะไม่อยู่ใน GitHub
4. ถ้าใช้ตำแหน่งอื่น ให้แก้ `PROJECT_DIR` ใน cell ถัดไป
5. เริ่มจาก `RUN_MODE = "smoke"`; เมื่อผลตรวจสอบถูกต้องจึงเปลี่ยนเป็น `"full"`


In [ ]:
# @title 1. Mount Google Drive and set parameters
from pathlib import Path
import os

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    default_project = Path('/content/drive/MyDrive/Data Analysis')
else:
    default_project = Path.cwd()

PROJECT_DIR = Path(os.environ.get('HT9046_PROJECT_DIR', str(default_project)))
DATA_ROOT = PROJECT_DIR
RUN_MODE = 'smoke'  # 'smoke' or 'full'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts' / f'shared_lstm_colab_{RUN_MODE}'

MODULE_IDS = [1, 2, 3, 4, 5, 6, 8]  # Module 7 excluded pending commissioning confirmation
MACHINE_DIRS = {
    'MX12': DATA_ROOT / 'Clean Data MX12',
    'MX25': DATA_ROOT / 'Clean Data MX25',
    'MX_007': DATA_ROOT / 'MX_007',
    'MX017': DATA_ROOT / 'MX017',
    'MX057': DATA_ROOT / 'MX057',
    'MX070': DATA_ROOT / 'MX070',
}

MAX_FILES_PER_MACHINE = 1 if RUN_MODE == 'smoke' else 7  # raise carefully; daily logs are large
MAX_TRAIN_WINDOWS_PER_GROUP = 500 if RUN_MODE == 'smoke' else 5000
MAX_VALID_WINDOWS_PER_GROUP = 150 if RUN_MODE == 'smoke' else 1000
MAX_TEST_WINDOWS_PER_GROUP = 150 if RUN_MODE == 'smoke' else 1000
EPOCHS = 2 if RUN_MODE == 'smoke' else 30
BATCH_SIZE = 128
MIN_WINDOWS_PER_GROUP = 30
RANDOM_SEED = 42

assert RUN_MODE in {'smoke', 'full'}
assert PROJECT_DIR.exists(), f'PROJECT_DIR not found: {PROJECT_DIR}'
print('Project:', PROJECT_DIR)
print('Artifacts:', ARTIFACT_DIR)
print('Mode:', RUN_MODE)


In [ ]:
# @title 2. Import project code and initialize runtime
import gc
import json
import re
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display

sys.path.insert(0, str(PROJECT_DIR))
from compressor_ml.anomaly import (
    StandardScaler3D, anomaly_score, health_score, pseudo_label,
    reconstruction_error,
)
from compressor_ml.config import PipelineConfig
from compressor_ml.features import engineer_features
from compressor_ml.model import build_lstm_autoencoder
from compressor_ml.preprocessing import (
    load_and_prepare, read_handler_log, validate_and_filter,
)
from compressor_ml.windowing import chronological_split, make_windows

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
CONFIG = PipelineConfig()
CONFIG.epochs = EPOCHS
CONFIG.batch_size = BATCH_SIZE
CONFIG.validate()

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Features:', len(CONFIG.feature_columns), '| Window rows:', CONFIG.window_rows)


## Context & Methods

### Key assumptions

- ข้อมูลไม่มี fault label จึงเป็น unsupervised anomaly detection ไม่ใช่การวินิจฉัย root cause หรือ RUL
- baseline ปกติของแต่ละ machine-module ต่างกัน จึง fit scaler เฉพาะจาก train split ของ group นั้น
- shared model เห็นข้อมูล normalized จากทุก group แต่ validation threshold ยังคงแยกต่อ group
- split 70/15/15 ตามเวลา **ภายในแต่ละ group** เพื่อป้องกันข้อมูลอนาคตรั่วเข้า train
- sample weight ทำให้แต่ละ group มีอิทธิพลรวมใกล้เคียงกัน แม้จำนวน windows ต่างกัน


In [ ]:
# @title 3. Discover bounded daily log files
DATE_PATTERN = re.compile(r'20\d{2}_\d{2}_\d{2}')

def discover_daily_files(folder: Path, max_files: int | None) -> list[Path]:
    if not folder.exists():
        return []
    candidates = []
    for pattern in ('*.csv', '*.txt'):
        for path in folder.rglob(pattern):
            name = path.name.lower()
            if 'static' in name or name.startswith('~$') or not DATE_PATTERN.search(path.name):
                continue
            candidates.append(path)
    files = sorted(set(candidates), key=lambda p: p.as_posix())
    return files[-max_files:] if max_files else files

selected_files_by_machine = {
    machine_id: discover_daily_files(folder, MAX_FILES_PER_MACHINE)
    for machine_id, folder in MACHINE_DIRS.items()
}
file_rows = [
    {'machine_id': machine_id, 'files': len(paths), 'latest_file': str(paths[-1]) if paths else None}
    for machine_id, paths in selected_files_by_machine.items()
]
file_table = pd.DataFrame(file_rows)
display(file_table)

available_machines = [m for m, paths in selected_files_by_machine.items() if paths]
if len(available_machines) < 2:
    raise ValueError('Shared-model smoke test needs data from at least two machines. Check PROJECT_DIR/MACHINE_DIRS.')


## Data preparation

แต่ละไฟล์ถูกอ่านครั้งเดียวต่อเครื่อง แล้วแยกประมวลผลเป็นราย module เพื่อประหยัดเวลา จากนั้นสร้าง 60-second windows โดยไม่ข้าม state transition หรือ time gap


In [ ]:
# @title 4. Build chronological datasets and per-group scalers
def even_take(values: np.ndarray, metadata: pd.DataFrame, limit: int):
    if limit is None or len(values) <= limit:
        return values, metadata.reset_index(drop=True)
    indices = np.linspace(0, len(values) - 1, limit, dtype=int)
    return values[indices], metadata.iloc[indices].reset_index(drop=True)

def safe_group_name(machine_id: str, module_id: int) -> str:
    safe_machine = re.sub(r'[^A-Za-z0-9_-]+', '_', machine_id)
    return f'{safe_machine}__M{module_id:02d}'

group_datasets = {}
group_scalers = {}
quality_rows = []
skipped_groups = []

for machine_id in available_machines:
    paths = selected_files_by_machine[machine_id]
    print(f'Loading {machine_id}: {len(paths)} file(s)')
    raw_machine = pd.concat(
        [read_handler_log(path, machine_id) for path in paths],
        ignore_index=True,
    )
    for module_id in MODULE_IDS:
        group_key = (machine_id, module_id)
        module_raw = raw_machine.loc[raw_machine['module_id'].eq(module_id)].copy()
        valid, rejected = validate_and_filter(module_raw, CONFIG)
        featured = engineer_features(valid, CONFIG)
        windows, window_meta = make_windows(featured, CONFIG)
        if len(windows) < MIN_WINDOWS_PER_GROUP:
            skipped_groups.append({'machine_id': machine_id, 'module_id': module_id, 'windows': len(windows)})
            continue

        (train_x, train_meta), (valid_x, valid_meta), (test_x, test_meta) = chronological_split(windows, window_meta)
        train_x, train_meta = even_take(train_x, train_meta, MAX_TRAIN_WINDOWS_PER_GROUP)
        valid_x, valid_meta = even_take(valid_x, valid_meta, MAX_VALID_WINDOWS_PER_GROUP)
        test_x, test_meta = even_take(test_x, test_meta, MAX_TEST_WINDOWS_PER_GROUP)

        scaler = StandardScaler3D().fit(train_x)
        group_scalers[group_key] = scaler
        group_datasets[group_key] = {
            'train': scaler.transform(train_x),
            'validation': scaler.transform(valid_x),
            'test': scaler.transform(test_x),
            'train_meta': train_meta,
            'validation_meta': valid_meta,
            'test_meta': test_meta,
        }
        quality_rows.append({
            'machine_id': machine_id, 'module_id': module_id,
            'accepted_rows': len(valid), 'rejected_rows': len(rejected),
            'all_windows': len(windows), 'train_windows': len(train_x),
            'validation_windows': len(valid_x), 'test_windows': len(test_x),
        })
    del raw_machine
    gc.collect()

quality_df = pd.DataFrame(quality_rows).sort_values(['machine_id', 'module_id']).reset_index(drop=True)
display(quality_df)
if skipped_groups:
    print('Skipped groups with too few windows:')
    display(pd.DataFrame(skipped_groups))
if len(group_datasets) < 2:
    raise ValueError('Need at least two valid machine-module groups after filtering.')


In [ ]:
# @title 5. Pool normalized windows with balanced group weights
train_parts = []
validation_parts = []
weight_parts = []

for group_key, dataset in group_datasets.items():
    group_train = dataset['train']
    train_parts.append(group_train)
    validation_parts.append(dataset['validation'])
    weight_parts.append(np.full((len(group_train), CONFIG.window_rows), 1.0 / len(group_train), dtype=np.float32))

pooled_train = np.concatenate(train_parts).astype(np.float32)
pooled_validation = np.concatenate(validation_parts).astype(np.float32)
sample_weights = np.concatenate(weight_parts)
sample_weights *= sample_weights.size / sample_weights.sum()

print('Valid groups:', len(group_datasets))
print('Pooled train:', pooled_train.shape)
print('Pooled validation:', pooled_validation.shape)
print('Sample-weight mean:', float(sample_weights.mean()))


## Shared model training

โมเดลส่วนกลางจะเรียนรู้รูปแบบ temporal ที่ใช้ร่วมกันจากข้อมูล normalized ทุก group ส่วน baseline เชิงตัวเลขยังถูกเก็บไว้ใน scaler ของแต่ละ group


In [ ]:
# @title 6. Train one shared LSTM Autoencoder
tf.keras.backend.clear_session()
shared_model = build_lstm_autoencoder(CONFIG, pooled_train.shape[-1])
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )
]
history = shared_model.fit(
    pooled_train, pooled_train,
    sample_weight=sample_weights,
    validation_data=(pooled_validation, pooled_validation),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    callbacks=callbacks,
    verbose=2,
)

pd.DataFrame(history.history).plot(figsize=(8, 4), title='Shared model training history')
plt.xlabel('Epoch')
plt.ylabel('MAE loss')
plt.grid(alpha=0.25)
plt.show()


In [ ]:
# @title 7. Calibrate a threshold and evaluate each group
group_thresholds = {}
metric_rows = []

for (machine_id, module_id), dataset in group_datasets.items():
    validation_pred = shared_model.predict(dataset['validation'], batch_size=BATCH_SIZE, verbose=0)
    validation_errors, _ = reconstruction_error(dataset['validation'], validation_pred)
    threshold = float(np.percentile(validation_errors, CONFIG.threshold_percentile))

    test_pred = shared_model.predict(dataset['test'], batch_size=BATCH_SIZE, verbose=0)
    test_errors, _ = reconstruction_error(dataset['test'], test_pred)
    safe_name = safe_group_name(machine_id, module_id)
    group_thresholds[safe_name] = {
        'machine_id': machine_id, 'module_id': module_id,
        'percentile': CONFIG.threshold_percentile, 'value': threshold,
    }
    metric_rows.append({
        'machine_id': machine_id, 'module_id': module_id,
        'validation_mae_p50': float(np.median(validation_errors)),
        'validation_mae_p99': threshold,
        'test_mae_p50': float(np.median(test_errors)),
        'test_mae_p95': float(np.percentile(test_errors, 95)),
        'test_exceedance_rate': float(np.mean(test_errors > threshold)),
    })

metrics_df = pd.DataFrame(metric_rows).sort_values(['machine_id', 'module_id']).reset_index(drop=True)
display(metrics_df.round(4))

ax = metrics_df.pivot(index='machine_id', columns='module_id', values='test_exceedance_rate').plot(
    kind='bar', figsize=(11, 4), title='Test threshold exceedance rate by machine-module'
)
ax.set_ylabel('Fraction above validation p99 threshold')
ax.axhline(0.10, color='red', linestyle='--', linewidth=1, label='Review level (10%)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# @title 8. Save the shared model and per-group calibration bundle
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
scaler_dir = ARTIFACT_DIR / 'scalers'
scaler_dir.mkdir(parents=True, exist_ok=True)

shared_model.save(ARTIFACT_DIR / 'shared_model.keras')
CONFIG.save(ARTIFACT_DIR / 'config.json')
for (machine_id, module_id), scaler in group_scalers.items():
    scaler.save(str(scaler_dir / f'{safe_group_name(machine_id, module_id)}.npz'))

with (ARTIFACT_DIR / 'thresholds.json').open('w', encoding='utf-8') as handle:
    json.dump(group_thresholds, handle, indent=2, ensure_ascii=False)
metrics_df.to_csv(ARTIFACT_DIR / 'group_metrics.csv', index=False)
quality_df.to_csv(ARTIFACT_DIR / 'data_quality_summary.csv', index=False)

manifest = {
    'model_type': 'shared_lstm_autoencoder_with_per_group_calibration',
    'model_version': f'shared_lstm_{RUN_MODE}_v1',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_mode': RUN_MODE,
    'groups': [safe_group_name(machine_id, module_id) for machine_id, module_id in group_datasets],
    'source_files': {m: [str(p) for p in paths] for m, paths in selected_files_by_machine.items()},
    'epochs_completed': len(history.history['loss']),
    'final_loss': float(history.history['loss'][-1]),
    'final_validation_loss': float(history.history['val_loss'][-1]),
    'threshold_method': f'per_group_validation_p{CONFIG.threshold_percentile:g}',
}
with (ARTIFACT_DIR / 'manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)

print('Saved artifact bundle to:', ARTIFACT_DIR)
print(sorted(path.name for path in ARTIFACT_DIR.iterdir()))


## Smoke inference on MX_007

cell ต่อไปใช้ shared model เดิม แต่เลือก scaler และ threshold ของ `MX_007 / Module 1` โดยอัตโนมัติ ใน smoke mode จะจำกัดข้อมูลล่าสุดไว้เพื่อลดเวลารัน


In [ ]:
# @title 9. Run state-aware inference for one machine-module
SCORE_MACHINE = 'MX_007'
SCORE_MODULE = 1
INFERENCE_ROWS_IN_SMOKE = 3600

def score_machine_module(machine_id: str, module_id: int, paths: list[Path]) -> pd.DataFrame:
    group_key = (machine_id, module_id)
    if group_key not in group_scalers:
        raise KeyError(f'No scaler/threshold available for {group_key}; inspect skipped_groups.')
    valid, _ = load_and_prepare(paths, machine_id, CONFIG, module_id)
    if RUN_MODE == 'smoke':
        valid = valid.tail(INFERENCE_ROWS_IN_SMOKE).copy()
    featured = engineer_features(valid, CONFIG)
    windows, window_meta = make_windows(featured, CONFIG)
    if len(windows) == 0:
        raise ValueError('No complete stable window for inference.')

    scaler = group_scalers[group_key]
    scaled = scaler.transform(windows)
    reconstructed = shared_model.predict(scaled, batch_size=BATCH_SIZE, verbose=0)
    errors, feature_errors = reconstruction_error(scaled, reconstructed)
    threshold = group_thresholds[safe_group_name(machine_id, module_id)]['value']
    scores = anomaly_score(errors, threshold)

    result = window_meta.copy()
    result['reconstruction_error'] = errors
    result['anomaly_score'] = scores
    result['health_score'] = result.groupby(
        ['machine_id', 'module_id', 'segment_id'], sort=False
    )['anomaly_score'].transform(
        lambda values: health_score(values.to_numpy(), CONFIG.health_smoothing_windows)
    )
    result['condition_status'] = [
        pseudo_label(value, CONFIG.normal_min, CONFIG.watch_min, CONFIG.warning_min)
        for value in result['health_score']
    ]
    top_indices = feature_errors.argmax(axis=1)
    result['top_error_feature'] = [CONFIG.feature_columns[index] for index in top_indices]
    result['threshold'] = threshold
    result['model_version'] = manifest['model_version']
    return result

score_paths = selected_files_by_machine[SCORE_MACHINE][-1:]
inference_df = score_machine_module(SCORE_MACHINE, SCORE_MODULE, score_paths)
inference_path = ARTIFACT_DIR / f'inference_{safe_group_name(SCORE_MACHINE, SCORE_MODULE)}.csv'
inference_df.to_csv(inference_path, index=False)

latest_columns = [
    'timestamp', 'reconstruction_error', 'anomaly_score', 'health_score',
    'condition_status', 'top_error_feature', 'threshold', 'model_version',
]
display(inference_df[latest_columns].tail(10))
print('Latest result:', inference_df[latest_columns].iloc[-1].to_dict())
print('Saved:', inference_path)


In [ ]:
# @title 10. Visual check of MX_007 health score
plot_frame = inference_df.tail(300).copy()
plot_frame['timestamp'] = pd.to_datetime(plot_frame['timestamp'])
ax = plot_frame.plot(
    x='timestamp', y='health_score', figsize=(12, 4),
    title=f'{SCORE_MACHINE} Module {SCORE_MODULE} — smoothed health score',
    legend=False,
)
for level, label, color in [
    (CONFIG.normal_min, 'Normal', 'green'),
    (CONFIG.watch_min, 'Watch', 'orange'),
    (CONFIG.warning_min, 'Warning', 'red'),
]:
    ax.axhline(level, linestyle='--', linewidth=1, color=color, label=label)
ax.set_ylim(0, 100)
ax.set_ylabel('Health score')
ax.legend()
plt.tight_layout()
plt.show()


## Checks & Next Steps

ก่อนเปลี่ยนเป็น full training ให้ตรวจสิ่งต่อไปนี้:

1. `quality_df` ต้องมี windows จากหลายเครื่องและไม่มีเครื่องใดหายไปโดยไม่ทราบสาเหตุ
2. ดู `test_exceedance_rate` แยก group; ค่าสูงกว่า 10% ควรตรวจ distribution shift, maintenance event หรือ baseline ที่ปน anomaly
3. Smoke result ใช้ยืนยันว่า pipeline รันครบเท่านั้น ห้ามใช้ตัดสินสภาพเครื่องหรือกำหนด maintenance action
4. Full run ควรใช้หลายวันต่อเครื่อง, `RUN_MODE='full'`, และตรวจผลแยกทุก machine-module
5. `condition_status` เป็น pseudo-label จาก reconstruction error ไม่ใช่ fault diagnosis และไม่ใช่ RUL
6. หากเพิ่มเครื่องใหม่ ให้เก็บ normal baseline เพื่อ fit scaler/threshold ใหม่ โดยยังใช้ shared model เดิมได้หลังผ่าน validation
